In [43]:
# ============================================================
# GLOBAL SPENDING, GROWTH, PROSPERITY, AND HAPPINESS ANALYSIS
# Data Cleaning and Preparation for Tableau
# Author: Ankita Tripathy
# ============================================================

# Project Purpose:
# This project analyzes how education spending, military spending,
# GDP per capita growth, prosperity, and happiness differ across countries
# by World Bank income group.

# This notebook cleans raw data from multiple sources and creates
# Tableau-ready CSV files.

# Final Tableau files created:
# 1. Income group.csv
# 2. GDP_Current_USD.csv
# 3. GDP_PC_Growth.csv
# 4. Education.csv
# 5. Military.csv
# 6. Happiness.csv
# 7. Prosperity.csv
# 8. Combined_Tableau_Analysis_2011_2021.csv

# Main analysis period:
# 2011 to 2021

In [44]:
# ============================================================
# STEP 0: INSTALL REQUIRED PACKAGES
# ============================================================

# Purpose:
# pandas is used for reading, cleaning, reshaping, and exporting data.
# numpy is used for numeric operations and missing values.
# openpyxl is used to read Excel .xlsx files.
# xlrd is used only if older .xls Excel files need to be read.

!pip install pandas numpy openpyxl xlrd

In [45]:
# ============================================================
# STEP 1: IMPORT REQUIRED LIBRARIES
# ============================================================

# Purpose:
# pathlib is used for clean file paths.
# re is used to extract year values from column names.
# pandas and numpy are used for all data cleaning work.

import re
import pandas as pd
import numpy as np
from pathlib import Path

In [46]:
# ============================================================
# STEP 2: SET PROJECT FOLDER PATHS
# ============================================================

# Purpose:
# The raw data folder stores the downloaded source files.
# The cleaned data folder stores the final CSV files for Tableau.

raw_folder = Path("/Users/ankitatripathy/Desktop/Tableau/Raw data")
cleaned_folder = Path("/Users/ankitatripathy/Desktop/Tableau/clean data")

# Create the cleaned folder if it does not already exist.
cleaned_folder.mkdir(parents=True, exist_ok=True)

print("Raw data folder:", raw_folder.resolve())
print("Cleaned data folder:", cleaned_folder.resolve())

Raw data folder: /Users/ankitatripathy/Desktop/Tableau/Raw data
Cleaned data folder: /Users/ankitatripathy/Desktop/Tableau/clean data


In [47]:
# ============================================================
# STEP 3: CHECK RAW FILES
# ============================================================

# Purpose:
# This confirms which files are present in the raw data folder.
# It helps prevent file path errors before cleaning starts.

print("Files available in raw data folder:")

for file in raw_folder.iterdir():
    print(file.name)

Files available in raw data folder:
Military_Current_USD_Raw.csv
.DS_Store
GDP_Per_Capita_Growth_Raw.csv
Prosperity_Raw.xlsx
Happiness_Raw.xlsx
Military_Pct_GDP_Raw.csv
Education_Pct_GDP_Raw.csv
GDP_Current_USD_Raw_Excel_Backup.xls
Income_Group_Raw.xlsx
GDP_Current_USD_Raw.csv


In [48]:
# ============================================================
# STEP 3: DEFINE RAW FILE NAMES
# ============================================================
# These names must match exactly with the files in the raw folder.

income_group_file = raw_folder / "Income_Group_Raw.xlsx"
happiness_file = raw_folder / "Happiness_Raw.xlsx"
gdp_current_file = raw_folder / "GDP_Current_USD_Raw.csv"
education_file = raw_folder / "Education_Pct_GDP_Raw.csv"
gdp_pc_growth_file = raw_folder / "GDP_Per_Capita_Growth_Raw.csv"
prosperity_file = raw_folder / "Prosperity_Raw.xlsx"

# Military data uses two World Bank files sourced from SIPRI indicators
military_current_usd_file = raw_folder / "Military_Current_USD_Raw.csv"
military_pct_gdp_file = raw_folder / "Military_Pct_GDP_Raw.csv"

raw_files = {
    "Income Group": income_group_file,
    "Happiness": happiness_file,
    "GDP Current USD": gdp_current_file,
    "Education % GDP": education_file,
    "GDP Per Capita Growth": gdp_pc_growth_file,
    "Prosperity": prosperity_file,
    "Military Current USD": military_current_usd_file,
    "Military % GDP": military_pct_gdp_file
}

print("Checking raw files:")
for name, path in raw_files.items():
    if path.exists():
        print(f"FOUND: {name} -> {path.name}")
    else:
        print(f"MISSING: {name} -> {path}")

Checking raw files:
FOUND: Income Group -> Income_Group_Raw.xlsx
FOUND: Happiness -> Happiness_Raw.xlsx
FOUND: GDP Current USD -> GDP_Current_USD_Raw.csv
FOUND: Education % GDP -> Education_Pct_GDP_Raw.csv
FOUND: GDP Per Capita Growth -> GDP_Per_Capita_Growth_Raw.csv
FOUND: Prosperity -> Prosperity_Raw.xlsx
FOUND: Military Current USD -> Military_Current_USD_Raw.csv
FOUND: Military % GDP -> Military_Pct_GDP_Raw.csv


In [49]:
# ============================================================
# STEP 4: CREATE HELPER FUNCTIONS
# ============================================================
# These functions keep the cleaning process consistent across files.

def clean_country_name(country):
    """
    Standardizes country names across datasets.
    Different data sources use different country names.
    This function reduces matching issues.
    """
    if pd.isna(country):
        return np.nan

    country = str(country).strip()
    country = re.sub(r"\s+", " ", country)

    country_mapping = {
        "United States of America": "United States",
        "USA": "United States",
        "Russian Federation": "Russia",
        "Korea, South": "Korea, Rep.",
        "South Korea": "Korea, Rep.",
        "Republic of Korea": "Korea, Rep.",
        "North Korea": "Korea, Dem. People's Rep.",
        "Iran": "Iran, Islamic Rep.",
        "Egypt": "Egypt, Arab Rep.",
        "Venezuela": "Venezuela, RB",
        "Yemen": "Yemen, Rep.",
        "Laos": "Lao PDR",
        "Vietnam": "Viet Nam",
        "Turkey": "Turkiye",
        "Türkiye": "Turkiye",
        "Czech Republic": "Czechia",
        "Bahamas": "Bahamas, The",
        "Gambia": "Gambia, The",
        "Micronesia (country)": "Micronesia, Fed. Sts."
    }

    return country_mapping.get(country, country)


def clean_country_code(code):
    """
    Cleans country codes.
    Country_Code is used as the main Tableau relationship key.
    """
    if pd.isna(code):
        return np.nan

    code = str(code).strip().upper()

    if code in ["", "NAN", "NA", "NONE"]:
        return np.nan

    return code


def clean_numeric(value):
    """
    Converts raw text values into numeric values.
    Removes commas, dollar signs, percent signs, and missing markers.
    """
    if pd.isna(value):
        return np.nan

    value = str(value).strip()
    value = value.replace(",", "")
    value = value.replace("$", "")
    value = value.replace("%", "")

    if value in ["", "..", "NA", "N/A", "nan", "NaN", "None", "-"]:
        return np.nan

    return pd.to_numeric(value, errors="coerce")


def extract_year(column_name):
    """
    Extracts a year from column names such as:
    2011
    2011 [YR2011]
    GDP_2011
    """
    match = re.search(r"(19|20)\d{2}", str(column_name))
    if match:
        return int(match.group(0))
    return None


def standardize_columns(df):
    """
    Standardizes common column names across sources.
    """
    df = df.copy()
    df.columns = df.columns.astype(str).str.strip()

    rename_map = {
        "Country Code": "Country_Code",
        "Code": "Country_Code",
        "ISO3": "Country_Code",
        "ISO Code": "Country_Code",

        "Country Name": "Standard_Country",
        "Country name": "Standard_Country",
        "Country": "Standard_Country",
        "Economy": "Standard_Country",
        "Entity": "Standard_Country",

        "Income group": "Income_Group",
        "Income Group": "Income_Group",
        "Region": "Region",

        "Year": "Year",
        "year": "Year"
    }

    df = df.rename(columns={col: rename_map.get(col, col) for col in df.columns})
    return df


def remove_aggregate_rows(df):
    """
    Removes aggregate rows such as World, OECD, High income, Low income, etc.
    These are not individual countries and can distort country-level analysis.
    """
    aggregate_codes = {
        "WLD", "HIC", "LIC", "LMC", "UMC", "LMY", "OED", "EUU",
        "EAS", "EAP", "ECA", "LAC", "MEA", "NAC", "SAS", "SSA",
        "SSF", "ARB", "IBD", "IBT", "IDA", "IDX", "CSS", "CEB",
        "EMU", "FCS", "HPC", "INX", "LDC", "MIC", "MNA", "OSS",
        "PRE", "PSS", "PST", "SST", "TEA", "TEC", "TLA", "TMN",
        "TSA", "TSS"
    }

    if "Country_Code" in df.columns:
        df = df[~df["Country_Code"].isin(aggregate_codes)]

    return df


def print_summary(df, name):
    """
    Prints a basic quality check for each cleaned dataset.
    """
    print("\n" + "=" * 80)
    print(name)
    print("=" * 80)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))

    if "Year" in df.columns:
        print("Year range:", df["Year"].min(), "to", df["Year"].max())

    if "Country_Code" in df.columns:
        print("Unique countries:", df["Country_Code"].nunique())

    print("\nMissing values:")
    print(df.isna().sum())

    print("\nPreview:")
    display(df.head())

In [50]:
# ============================================================
# STEP 5: CLEAN WORLD BANK INCOME GROUP DATA
# ============================================================
# Source file:
# Income_Group_Raw.xlsx
#
# Purpose:
# This file provides country code, country name, region,
# and World Bank income group.
#
# Tableau use:
# This becomes the central relationship table.

income_df = pd.read_excel(income_group_file)
income_df = standardize_columns(income_df)

print("Original Income Group columns:")
print(income_df.columns.tolist())

income_df = income_df.rename(columns={
    "Lending category": "Lending_Category",
    "Lending Category": "Lending_Category"
})

required_income_cols = ["Country_Code", "Standard_Country", "Region", "Income_Group"]

missing_cols = [col for col in required_income_cols if col not in income_df.columns]
if missing_cols:
    raise ValueError(f"Missing required columns in income group file: {missing_cols}")

income_df = income_df[required_income_cols].copy()

income_df["Country_Code"] = income_df["Country_Code"].apply(clean_country_code)
income_df["Standard_Country"] = income_df["Standard_Country"].apply(clean_country_name)

income_df = income_df.dropna(subset=["Country_Code", "Standard_Country", "Income_Group"])
income_df = income_df.drop_duplicates(subset=["Country_Code"])

income_output = cleaned_folder / "Income group.csv"
income_df.to_csv(income_output, index=False)

print("Saved:", income_output)
print_summary(income_df, "Income group.csv")

Original Income Group columns:
['Standard_Country', 'Country_Code', 'Region', 'Income_Group', 'Lending category']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/Income group.csv

Income group.csv
Shape: (216, 4)
Columns: ['Country_Code', 'Standard_Country', 'Region', 'Income_Group']
Unique countries: 216

Missing values:
Country_Code        0
Standard_Country    0
Region              0
Income_Group        0
dtype: int64

Preview:


,Country_Code,Standard_Country,Region,Income_Group
0,AFG,Afghanistan,"Middle East, North Africa, Afghanistan & Pakistan",Low income
1,ALB,Albania,Europe & Central Asia,Upper middle income
2,DZA,Algeria,"Middle East, North Africa, Afghanistan & Pakistan",Upper middle income
3,ASM,American Samoa,East Asia & Pacific,High income
4,AND,Andorra,Europe & Central Asia,High income


In [51]:
# ============================================================
# STEP 6: CLEAN WORLD BANK WIDE CSV FILES
# ============================================================
# Used for:
# GDP current US$
# Education spending as % of GDP
# GDP per capita growth annual %
#
# World Bank CSV files usually have 4 metadata rows before data starts.
# This function skips those rows, reshapes year columns into rows,
# and creates Tableau-ready long format.

def clean_world_bank_csv(file_path, value_column_name):
    """
    Reads a World Bank CSV file and converts it from wide format to long format.

    Input format:
    Country Name | Country Code | Indicator Name | Indicator Code | 1960 | 1961 ...

    Output format:
    Country_Code | Standard_Country | Year | value_column_name
    """
    df = pd.read_csv(file_path, skiprows=4)
    df = standardize_columns(df)

    print(f"\nOriginal columns in {file_path.name}:")
    print(df.columns.tolist())

    required_cols = ["Country_Code", "Standard_Country"]

    missing_cols = [col for col in required_cols if col not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing columns in {file_path.name}: {missing_cols}")

    year_cols = [col for col in df.columns if extract_year(col) is not None]

    if len(year_cols) == 0:
        raise ValueError(f"No year columns found in {file_path.name}")

    df = df[["Country_Code", "Standard_Country"] + year_cols].copy()

    df_long = df.melt(
        id_vars=["Country_Code", "Standard_Country"],
        value_vars=year_cols,
        var_name="Year",
        value_name=value_column_name
    )

    df_long["Year"] = df_long["Year"].apply(extract_year)
    df_long[value_column_name] = df_long[value_column_name].apply(clean_numeric)

    df_long["Country_Code"] = df_long["Country_Code"].apply(clean_country_code)
    df_long["Standard_Country"] = df_long["Standard_Country"].apply(clean_country_name)

    df_long = remove_aggregate_rows(df_long)
    df_long = df_long.dropna(subset=["Country_Code", "Standard_Country", "Year"])
    df_long = df_long.drop_duplicates(subset=["Country_Code", "Year"])

    return df_long

In [52]:
# ============================================================
# STEP 7: CLEAN GDP CURRENT USD DATA
# ============================================================
# Source file:
# GDP_Current_USD_Raw.csv
#
# Indicator:
# GDP current US$
#
# Purpose:
# Measures total economic size by country and year.

gdp_current_df = clean_world_bank_csv(
    gdp_current_file,
    "GDP_Current_USD"
)

gdp_current_output = cleaned_folder / "GDP_Current_USD.csv"
gdp_current_df.to_csv(gdp_current_output, index=False)

print("Saved:", gdp_current_output)
print_summary(gdp_current_df, "GDP_Current_USD.csv")


Original columns in GDP_Current_USD_Raw.csv:
['Standard_Country', 'Country_Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/GDP_Current_USD.csv

GDP_Current_USD.csv
Shape: (14784, 4)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'GDP_Current_USD']
Year range: 1960 to 2025
Unique countries: 224

Missing values:
Country_Code           0
Standard_Country       0
Year                   0
GDP_Current

,Country_Code,Standard_Country,Year,GDP_Current_USD
0,ABW,Aruba,1960,NaN
1,AFE,Africa Eastern and Southern,1960,2.420569e+10
2,AFG,Afghanistan,1960,NaN
3,AFW,Africa Western and Central,1960,1.190481e+10
4,AGO,Angola,1960,NaN


In [53]:
# ============================================================
# STEP 8: CLEAN EDUCATION SPENDING DATA
# ============================================================
# Source file:
# Education_Pct_GDP_Raw.csv
#
# Indicator:
# Government expenditure on education, total (% of GDP)
#
# Purpose:
# Measures how much each country spends on education as a share of GDP.

education_df = clean_world_bank_csv(
    education_file,
    "Education_Pct_GDP"
)

education_output = cleaned_folder / "Education.csv"
education_df.to_csv(education_output, index=False)

print("Saved:", education_output)
print_summary(education_df, "Education.csv")


Original columns in Education_Pct_GDP_Raw.csv:
['Standard_Country', 'Country_Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/Education.csv

Education.csv
Shape: (14784, 4)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'Education_Pct_GDP']
Year range: 1960 to 2025
Unique countries: 224

Missing values:
Country_Code            0
Standard_Country        0
Year                    0
Education_Pct_GD

,Country_Code,Standard_Country,Year,Education_Pct_GDP
0,ABW,Aruba,1960,NaN
1,AFE,Africa Eastern and Southern,1960,NaN
2,AFG,Afghanistan,1960,NaN
3,AFW,Africa Western and Central,1960,NaN
4,AGO,Angola,1960,NaN


In [54]:
# ============================================================
# STEP 9: CLEAN GDP PER CAPITA GROWTH DATA
# ============================================================
# Source file:
# GDP_Per_Capita_Growth_Raw.csv
#
# Indicator:
# GDP per capita growth annual %
#
# Purpose:
# Measures annual economic growth per person.

gdp_growth_df = clean_world_bank_csv(
    gdp_pc_growth_file,
    "GDP_Per_Capita_Growth_Pct"
)

gdp_growth_output = cleaned_folder / "GDP_PC_Growth.csv"
gdp_growth_df.to_csv(gdp_growth_output, index=False)

print("Saved:", gdp_growth_output)
print_summary(gdp_growth_df, "GDP_PC_Growth.csv")


Original columns in GDP_Per_Capita_Growth_Raw.csv:
['Standard_Country', 'Country_Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/GDP_PC_Growth.csv

GDP_PC_Growth.csv
Shape: (14784, 4)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'GDP_Per_Capita_Growth_Pct']
Year range: 1960 to 2025
Unique countries: 224

Missing values:
Country_Code                    0
Standard_Country                0
Year  

,Country_Code,Standard_Country,Year,GDP_Per_Capita_Growth_Pct
0,ABW,Aruba,1960,NaN
1,AFE,Africa Eastern and Southern,1960,NaN
2,AFG,Afghanistan,1960,NaN
3,AFW,Africa Western and Central,1960,NaN
4,AGO,Angola,1960,NaN


In [55]:
# ============================================================
# STEP 10: CLEAN MILITARY SPENDING DATA
# ============================================================
# Source files:
# 1. Military_Current_USD_Raw.csv
# 2. Military_Pct_GDP_Raw.csv
#
# Source:
# World Bank military expenditure indicators
# Original source: SIPRI Military Expenditure Database
#
# Purpose:
# This step creates one military dataset with:
# Country_Code
# Standard_Country
# Year
# Military_Exp_USD
# Military_Pct_GDP

military_current_usd_file = raw_folder / "Military_Current_USD_Raw.csv"
military_pct_gdp_file = raw_folder / "Military_Pct_GDP_Raw.csv"

# Clean military expenditure in current US dollars
military_usd_df = clean_world_bank_csv(
    military_current_usd_file,
    "Military_Exp_USD"
)

# Clean military expenditure as % of GDP
military_pct_df = clean_world_bank_csv(
    military_pct_gdp_file,
    "Military_Pct_GDP"
)

# Merge both military datasets using country and year
military_df = pd.merge(
    military_usd_df,
    military_pct_df,
    on=["Country_Code", "Standard_Country", "Year"],
    how="outer"
)

# Keep only the final Tableau-ready columns
military_df = military_df[
    ["Country_Code", "Standard_Country", "Year", "Military_Exp_USD", "Military_Pct_GDP"]
].copy()

# Final cleaning
military_df["Country_Code"] = military_df["Country_Code"].apply(clean_country_code)
military_df["Standard_Country"] = military_df["Standard_Country"].apply(clean_country_name)
military_df["Year"] = pd.to_numeric(military_df["Year"], errors="coerce").astype("Int64")
military_df["Military_Exp_USD"] = military_df["Military_Exp_USD"].apply(clean_numeric)
military_df["Military_Pct_GDP"] = military_df["Military_Pct_GDP"].apply(clean_numeric)

military_df = remove_aggregate_rows(military_df)
military_df = military_df.dropna(subset=["Country_Code", "Standard_Country", "Year"])
military_df = military_df.drop_duplicates(subset=["Country_Code", "Year"])

# Save final military file
military_output = cleaned_folder / "Military.csv"
military_df.to_csv(military_output, index=False)

print("Saved:", military_output)
print_summary(military_df, "Military.csv")


Original columns in Military_Current_USD_Raw.csv:
['Standard_Country', 'Country_Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986', '1987', '1988', '1989', '1990', '1991', '1992', '1993', '1994', '1995', '1996', '1997', '1998', '1999', '2000', '2001', '2002', '2003', '2004', '2005', '2006', '2007', '2008', '2009', '2010', '2011', '2012', '2013', '2014', '2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024', '2025', 'Unnamed: 70']

Original columns in Military_Pct_GDP_Raw.csv:
['Standard_Country', 'Country_Code', 'Indicator Name', 'Indicator Code', '1960', '1961', '1962', '1963', '1964', '1965', '1966', '1967', '1968', '1969', '1970', '1971', '1972', '1973', '1974', '1975', '1976', '1977', '1978', '1979', '1980', '1981', '1982', '1983', '1984', '1985', '1986

,Country_Code,Standard_Country,Year,Military_Exp_USD,Military_Pct_GDP
0,ABW,Aruba,1960,NaN,NaN
1,ABW,Aruba,1961,NaN,NaN
2,ABW,Aruba,1962,NaN,NaN
3,ABW,Aruba,1963,NaN,NaN
4,ABW,Aruba,1964,NaN,NaN


In [56]:
# ============================================================
# STEP 11: CLEAN WORLD HAPPINESS REPORT DATA
# ============================================================
# Source file:
# Happiness_Raw.xlsx
#
# Purpose:
# This step cleans the World Happiness Report data.
# It creates Happiness.csv for Tableau.
#
# Main field used:
# Life evaluation (3-year average), renamed as Happiness_Score.
#
# Additional fields kept:
# Log GDP per capita
# Social support
# Healthy life expectancy
# Freedom
# Generosity
# Perceptions of corruption
# Dystopia residual

happiness_raw = pd.read_excel(happiness_file)
happiness_raw = standardize_columns(happiness_raw)

print("Original Happiness columns:")
print(happiness_raw.columns.tolist())

# Rename World Happiness Report columns into simpler project-friendly names
happiness_raw = happiness_raw.rename(columns={
    "Life evaluation (3-year average)": "Happiness_Score",
    "Life Ladder": "Happiness_Score",
    "Ladder score": "Happiness_Score",
    "Happiness score": "Happiness_Score",
    "Happiness Score": "Happiness_Score",

    "Explained by: Log GDP per capita": "Log_GDP_per_Capita",
    "Explained by: Social support": "Social_Support",
    "Explained by: Healthy life expectancy": "Healthy_Life_Expectancy",
    "Explained by: Freedom to make life choices": "Freedom",
    "Explained by: Generosity": "Generosity",
    "Explained by: Perceptions of corruption": "Corruption",
    "Dystopia + residual": "Dystopia_Residual"
})

# Check required columns
required_happiness_cols = ["Standard_Country", "Year", "Happiness_Score"]

missing_cols = [col for col in required_happiness_cols if col not in happiness_raw.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in happiness file: {missing_cols}")

# Standardize country names
happiness_raw["Standard_Country"] = happiness_raw["Standard_Country"].apply(clean_country_name)

# Add Country_Code using Income Group table
country_code_map = income_df.set_index("Standard_Country")["Country_Code"].to_dict()
happiness_raw["Country_Code"] = happiness_raw["Standard_Country"].map(country_code_map)

# Keep only useful columns for Tableau
happiness_columns = [
    "Country_Code",
    "Standard_Country",
    "Year",
    "Happiness_Score",
    "Log_GDP_per_Capita",
    "Social_Support",
    "Healthy_Life_Expectancy",
    "Freedom",
    "Generosity",
    "Corruption",
    "Dystopia_Residual"
]

# If any optional column is missing, create it as blank
for col in happiness_columns:
    if col not in happiness_raw.columns:
        happiness_raw[col] = np.nan

happiness_df = happiness_raw[happiness_columns].copy()

# Clean final columns
happiness_df["Country_Code"] = happiness_df["Country_Code"].apply(clean_country_code)
happiness_df["Standard_Country"] = happiness_df["Standard_Country"].apply(clean_country_name)
happiness_df["Year"] = pd.to_numeric(happiness_df["Year"], errors="coerce").astype("Int64")

# Convert numeric fields
for col in happiness_columns:
    if col not in ["Country_Code", "Standard_Country", "Year"]:
        happiness_df[col] = happiness_df[col].apply(clean_numeric)

# Remove rows that cannot be linked in Tableau
happiness_df = happiness_df.dropna(subset=["Country_Code", "Standard_Country", "Year"])

# Remove duplicate country-year records
happiness_df = happiness_df.drop_duplicates(subset=["Country_Code", "Year"])

# Save cleaned file
happiness_output = cleaned_folder / "Happiness.csv"
happiness_df.to_csv(happiness_output, index=False)

print("Saved:", happiness_output)
print_summary(happiness_df, "Happiness.csv")

Original Happiness columns:
['Year', 'Rank', 'Standard_Country', 'Life evaluation (3-year average)', 'Lower whisker', 'Upper whisker', 'Explained by: Log GDP per capita', 'Explained by: Social support', 'Explained by: Healthy life expectancy', 'Explained by: Freedom to make life choices', 'Explained by: Generosity', 'Explained by: Perceptions of corruption', 'Dystopia + residual']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/Happiness.csv

Happiness.csv
Shape: (1945, 11)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'Happiness_Score', 'Log_GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Generosity', 'Corruption', 'Dystopia_Residual']
Year range: 2011 to 2025
Unique countries: 152

Missing values:
Country_Code                  0
Standard_Country              0
Year                          0
Happiness_Score               0
Log_GDP_per_Capita         1000
Social_Support             1000
Healthy_Life_Expectancy    1000
Freedom                  

,Country_Code,Standard_Country,Year,Happiness_Score,Log_GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Generosity,Corruption,Dystopia_Residual
0,FIN,Finland,2025,7.764,1.915,1.638,0.939,1.105,0.093,0.491,1.582
1,ISL,Iceland,2025,7.540,1.971,1.720,0.996,1.105,0.187,0.187,1.373
2,DNK,Denmark,2025,7.539,1.986,1.633,0.930,1.081,0.125,0.474,1.310
3,CRI,Costa Rica,2025,7.439,1.697,1.483,0.739,1.101,0.059,0.122,2.236
4,SWE,Sweden,2025,7.255,1.950,1.570,1.027,1.070,0.149,0.447,1.041


In [57]:
# ============================================================
# STEP 12: CLEAN PROSPERITY DATA
# ============================================================
# Source file:
# Prosperity_Raw.xlsx
#
# Purpose:
# This step cleans the Prosperity Index data.
# It creates Prosperity.csv for Tableau.
#
# Final fields:
# Country_Code
# Standard_Country
# Year
# Prosperity_Score

prosperity_raw = pd.read_excel(prosperity_file)
prosperity_raw = standardize_columns(prosperity_raw)

print("Original Prosperity columns:")
print(prosperity_raw.columns.tolist())

# Rename only the prosperity score column
# Do NOT rename Country Prosperity to Standard_Country because Standard_Country already exists.
prosperity_raw = prosperity_raw.rename(columns={
    "Prosperity score": "Prosperity_Score",
    "Prosperity Score": "Prosperity_Score",
    "Overall Prosperity": "Prosperity_Score"
})

# If Standard_Country is missing, use Country Prosperity as the country name
if "Standard_Country" not in prosperity_raw.columns and "Country Prosperity" in prosperity_raw.columns:
    prosperity_raw["Standard_Country"] = prosperity_raw["Country Prosperity"]

# If Country_Code is missing but Code exists, use Code as Country_Code
if "Country_Code" not in prosperity_raw.columns and "Code" in prosperity_raw.columns:
    prosperity_raw["Country_Code"] = prosperity_raw["Code"]

# Check required columns
required_prosperity_cols = ["Country_Code", "Standard_Country", "Year", "Prosperity_Score"]

missing_cols = [col for col in required_prosperity_cols if col not in prosperity_raw.columns]

if missing_cols:
    raise ValueError(f"Missing required columns in prosperity file: {missing_cols}")

# Keep final Tableau-ready columns
prosperity_df = prosperity_raw[
    ["Country_Code", "Standard_Country", "Year", "Prosperity_Score"]
].copy()

# Remove duplicate columns if any still exist
prosperity_df = prosperity_df.loc[:, ~prosperity_df.columns.duplicated()].copy()

# Clean final columns
prosperity_df["Country_Code"] = prosperity_df["Country_Code"].apply(clean_country_code)
prosperity_df["Standard_Country"] = prosperity_df["Standard_Country"].apply(clean_country_name)
prosperity_df["Year"] = pd.to_numeric(prosperity_df["Year"], errors="coerce").astype("Int64")
prosperity_df["Prosperity_Score"] = prosperity_df["Prosperity_Score"].apply(clean_numeric)

# Remove incomplete key rows
prosperity_df = prosperity_df.dropna(subset=["Country_Code", "Standard_Country", "Year"])

# Remove duplicate country-year records
prosperity_df = prosperity_df.drop_duplicates(subset=["Country_Code", "Year"])

# Save cleaned file
prosperity_output = cleaned_folder / "Prosperity.csv"
prosperity_df.to_csv(prosperity_output, index=False)

print("Saved:", prosperity_output)
print_summary(prosperity_df, "Prosperity.csv")

Original Prosperity columns:
['Country Prosperity', 'Country_Code', 'Year', 'Prosperity Score', 'Standard_Country']
Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/Prosperity.csv

Prosperity.csv
Shape: (2839, 4)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'Prosperity_Score']
Year range: 2007 to 2023
Unique countries: 167

Missing values:
Country_Code        0
Standard_Country    0
Year                0
Prosperity_Score    0
dtype: int64

Preview:


,Country_Code,Standard_Country,Year,Prosperity_Score
0,AFG,Afghanistan,2007,35.133427
1,ALB,Albania,2007,55.708755
2,DZA,Algeria,2007,48.161085
3,AGO,Angola,2007,37.322334
4,ARG,Argentina,2007,58.635379


In [58]:
# ============================================================
# STEP 13: CREATE COMBINED VALIDATION DATASET
# ============================================================
# Purpose:
# Tableau uses the 7 separate CSV files.
# This combined file is created only for validation and GitHub documentation.
#
# The merge is based mainly on Country_Code and Year because
# Country_Code is the official relationship key used in Tableau.

country_year_base = pd.concat([
    gdp_current_df[["Country_Code", "Standard_Country", "Year"]],
    gdp_growth_df[["Country_Code", "Standard_Country", "Year"]],
    education_df[["Country_Code", "Standard_Country", "Year"]],
    military_df[["Country_Code", "Standard_Country", "Year"]],
    happiness_df[["Country_Code", "Standard_Country", "Year"]],
    prosperity_df[["Country_Code", "Standard_Country", "Year"]]
], ignore_index=True)

country_year_base = country_year_base.drop_duplicates(subset=["Country_Code", "Year"])

combined_df = country_year_base.merge(
    income_df[["Country_Code", "Region", "Income_Group"]],
    on="Country_Code",
    how="left"
)

combined_df = combined_df.merge(
    gdp_current_df[["Country_Code", "Year", "GDP_Current_USD"]],
    on=["Country_Code", "Year"],
    how="left"
)

combined_df = combined_df.merge(
    gdp_growth_df[["Country_Code", "Year", "GDP_Per_Capita_Growth_Pct"]],
    on=["Country_Code", "Year"],
    how="left"
)

combined_df = combined_df.merge(
    education_df[["Country_Code", "Year", "Education_Pct_GDP"]],
    on=["Country_Code", "Year"],
    how="left"
)

combined_df = combined_df.merge(
    military_df[["Country_Code", "Year", "Military_Exp_USD", "Military_Pct_GDP"]],
    on=["Country_Code", "Year"],
    how="left"
)

combined_df = combined_df.merge(
    happiness_df[
        [
            "Country_Code", "Year", "Happiness_Score",
            "Log_GDP_per_Capita", "Social_Support",
            "Healthy_Life_Expectancy", "Freedom",
            "Generosity", "Corruption", "Dystopia_Residual"
        ]
    ],
    on=["Country_Code", "Year"],
    how="left"
)

combined_df = combined_df.merge(
    prosperity_df[["Country_Code", "Year", "Prosperity_Score"]],
    on=["Country_Code", "Year"],
    how="left"
)

print_summary(combined_df, "Combined Dataset Before Filtering")


Combined Dataset Before Filtering
Shape: (14801, 19)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'Region', 'Income_Group', 'GDP_Current_USD', 'GDP_Per_Capita_Growth_Pct', 'Education_Pct_GDP', 'Military_Exp_USD', 'Military_Pct_GDP', 'Happiness_Score', 'Log_GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Generosity', 'Corruption', 'Dystopia_Residual', 'Prosperity_Score']
Year range: 1960 to 2025
Unique countries: 225

Missing values:
Country_Code                     0
Standard_Country                 0
Year                             0
Region                         594
Income_Group                   594
GDP_Current_USD               2777
GDP_Per_Capita_Growth_Pct     3139
Education_Pct_GDP             9506
Military_Exp_USD              6159
Military_Pct_GDP              6476
Happiness_Score              12856
Log_GDP_per_Capita           13856
Social_Support               13856
Healthy_Life_Expectancy      13856
Freedom                      13858
Ge

,Country_Code,Standard_Country,Year,Region,Income_Group,GDP_Current_USD,GDP_Per_Capita_Growth_Pct,Education_Pct_GDP,Military_Exp_USD,Military_Pct_GDP,Happiness_Score,Log_GDP_per_Capita,Social_Support,Healthy_Life_Expectancy,Freedom,Generosity,Corruption,Dystopia_Residual,Prosperity_Score
0,ABW,Aruba,1960,Latin America & Caribbean,High income,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFE,Africa Eastern and Southern,1960,NaN,NaN,2.420569e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Afghanistan,1960,"Middle East, North Africa, Afghanistan & Pakistan",Low income,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFW,Africa Western and Central,1960,NaN,NaN,1.190481e+10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AGO,Angola,1960,Sub-Saharan Africa,Lower middle income,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [59]:
# ============================================================
# STEP 14: CREATE DERIVED FIELDS
# ============================================================
# Purpose:
# These fields support Tableau analysis and validation.
#
# These are the same types of calculated fields used in Tableau:
# GDP in billions
# GDP in trillions
# Education-military spending gap
# Spending priority category
# Data completeness flags

# Convert GDP into billions and trillions for easier visualization
combined_df["GDP_in_Billions"] = combined_df["GDP_Current_USD"] / 1_000_000_000
combined_df["GDP_in_Trillions"] = combined_df["GDP_Current_USD"] / 1_000_000_000_000

# Education-military gap:
# Positive value means education spending is higher than military spending.
# Negative value means military spending is higher than education spending.
combined_df["Education_Military_Gap"] = (
    combined_df["Education_Pct_GDP"] - combined_df["Military_Pct_GDP"]
)

# Spending priority:
# This creates a simple category showing whether a country is more education-focused,
# military-focused, balanced, or missing data.
combined_df["Spending_Priority"] = np.select(
    [
        combined_df["Education_Pct_GDP"].isna() | combined_df["Military_Pct_GDP"].isna(),
        combined_df["Education_Pct_GDP"] > combined_df["Military_Pct_GDP"],
        combined_df["Military_Pct_GDP"] > combined_df["Education_Pct_GDP"]
    ],
    [
        "Data Not Available",
        "Education-Focused",
        "Military-Focused"
    ],
    default="Balanced"
)

# Complete core data flag:
# Used to identify rows that have all main spending and growth fields.
combined_df["Complete_Core_Data_Flag"] = np.where(
    combined_df[
        ["Education_Pct_GDP", "Military_Pct_GDP", "GDP_Per_Capita_Growth_Pct"]
    ].notna().all(axis=1),
    "Complete Core Data",
    "Missing Core Data"
)

# Complete social data flag:
# Used to identify rows that have education, happiness, and prosperity fields.
combined_df["Complete_Social_Data_Flag"] = np.where(
    combined_df[
        ["Education_Pct_GDP", "Happiness_Score", "Prosperity_Score"]
    ].notna().all(axis=1),
    "Complete Social Data",
    "Missing Social Data"
)

display(combined_df.head())

,Country_Code,Standard_Country,Year,Region,Income_Group,GDP_Current_USD,GDP_Per_Capita_Growth_Pct,Education_Pct_GDP,Military_Exp_USD,Military_Pct_GDP,...,Generosity,Corruption,Dystopia_Residual,Prosperity_Score,GDP_in_Billions,GDP_in_Trillions,Education_Military_Gap,Spending_Priority,Complete_Core_Data_Flag,Complete_Social_Data_Flag
0,ABW,Aruba,1960,Latin America & Caribbean,High income,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data Not Available,Missing Core Data,Missing Social Data
1,AFE,Africa Eastern and Southern,1960,NaN,NaN,2.420569e+10,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,24.205689,0.024206,NaN,Data Not Available,Missing Core Data,Missing Social Data
2,AFG,Afghanistan,1960,"Middle East, North Africa, Afghanistan & Pakistan",Low income,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data Not Available,Missing Core Data,Missing Social Data
3,AFW,Africa Western and Central,1960,NaN,NaN,1.190481e+10,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,11.904806,0.011905,NaN,Data Not Available,Missing Core Data,Missing Social Data
4,AGO,Angola,1960,Sub-Saharan Africa,Lower middle income,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Data Not Available,Missing Core Data,Missing Social Data


In [60]:
# ============================================================
# STEP 15: FILTER MAIN ANALYSIS PERIOD
# ============================================================
# Purpose:
# The final Tableau dashboards use 2011 to 2021.
#
# Why this period:
# It gives the most consistent overlap across education spending,
# military spending, GDP per capita growth, prosperity, and happiness data.

analysis_df = combined_df[
    (combined_df["Year"] >= 2011) &
    (combined_df["Year"] <= 2021)
].copy()

combined_output = cleaned_folder / "Combined_Tableau_Analysis_2011_2021.csv"
analysis_df.to_csv(combined_output, index=False)

print("Saved:", combined_output)
print_summary(analysis_df, "Combined_Tableau_Analysis_2011_2021.csv")

Saved: /Users/ankitatripathy/Desktop/Tableau/clean data/Combined_Tableau_Analysis_2011_2021.csv

Combined_Tableau_Analysis_2011_2021.csv
Shape: (2475, 25)
Columns: ['Country_Code', 'Standard_Country', 'Year', 'Region', 'Income_Group', 'GDP_Current_USD', 'GDP_Per_Capita_Growth_Pct', 'Education_Pct_GDP', 'Military_Exp_USD', 'Military_Pct_GDP', 'Happiness_Score', 'Log_GDP_per_Capita', 'Social_Support', 'Healthy_Life_Expectancy', 'Freedom', 'Generosity', 'Corruption', 'Dystopia_Residual', 'Prosperity_Score', 'GDP_in_Billions', 'GDP_in_Trillions', 'Education_Military_Gap', 'Spending_Priority', 'Complete_Core_Data_Flag', 'Complete_Social_Data_Flag']
Year range: 2011 to 2021
Unique countries: 225

Missing values:
Country_Code                    0
Standard_Country                0
Year                            0
Region                         99
Income_Group                   99
GDP_Current_USD                71
GDP_Per_Capita_Growth_Pct      88
Education_Pct_GDP             623
Military_Exp

,Country_Code,Standard_Country,Year,Region,Income_Group,GDP_Current_USD,GDP_Per_Capita_Growth_Pct,Education_Pct_GDP,Military_Exp_USD,Military_Pct_GDP,...,Generosity,Corruption,Dystopia_Residual,Prosperity_Score,GDP_in_Billions,GDP_in_Trillions,Education_Military_Gap,Spending_Priority,Complete_Core_Data_Flag,Complete_Social_Data_Flag
11424,ABW,Aruba,2011,Latin America & Caribbean,High income,2.637859e+09,2.610525,5.914670,NaN,NaN,...,NaN,NaN,NaN,NaN,2.637859,0.002638,NaN,Data Not Available,Missing Core Data,Missing Social Data
11425,AFE,Africa Eastern and Southern,2011,NaN,NaN,9.598225e+11,1.320563,4.639015,1.324343e+10,1.525527,...,NaN,NaN,NaN,NaN,959.822476,0.959822,3.113488,Education-Focused,Complete Core Data,Missing Social Data
11426,AFG,Afghanistan,2011,"Middle East, North Africa, Afghanistan & Pakistan",Low income,1.780510e+10,-3.213295,3.462010,3.258069e+08,1.821345,...,NaN,NaN,NaN,34.871344,17.805098,0.017805,1.640664,Education-Focused,Complete Core Data,Complete Social Data
11427,AFW,Africa Western and Central,2011,NaN,NaN,6.911334e+11,2.001657,2.653360,4.987983e+09,0.800119,...,NaN,NaN,NaN,NaN,691.133400,0.691133,1.853241,Education-Focused,Complete Core Data,Missing Social Data
11428,AGO,Angola,2011,Sub-Saharan Africa,Lower middle income,1.255516e+11,-0.356512,NaN,3.639496e+09,3.255660,...,NaN,NaN,NaN,39.857896,125.551635,0.125552,NaN,Data Not Available,Missing Core Data,Missing Social Data


In [61]:
# ============================================================
# STEP 16: FINAL DATA QUALITY CHECK
# ============================================================
# Purpose:
# This step checks whether all cleaned files were created correctly.
# It also checks the final row counts, year range, and missing values.

print("Final cleaned dataset shapes:")
print("Income group:", income_df.shape)
print("GDP Current USD:", gdp_current_df.shape)
print("GDP Per Capita Growth:", gdp_growth_df.shape)
print("Education:", education_df.shape)
print("Military:", military_df.shape)
print("Happiness:", happiness_df.shape)
print("Prosperity:", prosperity_df.shape)
print("Combined Analysis:", analysis_df.shape)

print("\nFinal cleaned files created:")
for file in cleaned_folder.iterdir():
    if file.suffix == ".csv":
        print(file.name)

print("\nYear range in final combined dataset:")
print(analysis_df["Year"].min(), "to", analysis_df["Year"].max())

print("\nMissing values in key analysis fields:")

key_fields = [
    "Education_Pct_GDP",
    "Military_Pct_GDP",
    "GDP_Per_Capita_Growth_Pct",
    "Happiness_Score",
    "Prosperity_Score"
]

print(analysis_df[key_fields].isna().sum())

Final cleaned dataset shapes:
Income group: (216, 4)
GDP Current USD: (14784, 4)
GDP Per Capita Growth: (14784, 4)
Education: (14784, 4)
Military: (14784, 5)
Happiness: (1945, 11)
Prosperity: (2839, 4)
Combined Analysis: (2475, 25)

Final cleaned files created:
Prosperity.csv
Education.csv
Military.csv
Combined_Tableau_Analysis_2011_2021.csv
Happiness.csv
GDP_PC_Growth.csv
GDP_Current_USD.csv
Income group.csv

Year range in final combined dataset:
2011 to 2021

Missing values in key analysis fields:
Education_Pct_GDP             623
Military_Pct_GDP              730
GDP_Per_Capita_Growth_Pct      88
Happiness_Score              1062
Prosperity_Score              638
dtype: int64


In [62]:
# ============================================================
# STEP 17: FINAL PROJECT SUMMARY
# ============================================================
# Purpose:
# This summary documents the final output of the cleaning notebook.

print("Data cleaning completed successfully.")

print("\nUse these files in Tableau:")
print("1. Income group.csv")
print("2. GDP_Current_USD.csv")
print("3. GDP_PC_Growth.csv")
print("4. Education.csv")
print("5. Military.csv")
print("6. Happiness.csv")
print("7. Prosperity.csv")

print("\nUse Income group.csv as the central table.")
print("Connect the other six files using Country_Code.")

print("\nMain analysis period:")
print("2011 to 2021")

print("\nOptional validation file:")
print("Combined_Tableau_Analysis_2011_2021.csv")

Data cleaning completed successfully.

Use these files in Tableau:
1. Income group.csv
2. GDP_Current_USD.csv
3. GDP_PC_Growth.csv
4. Education.csv
5. Military.csv
6. Happiness.csv
7. Prosperity.csv

Use Income group.csv as the central table.
Connect the other six files using Country_Code.

Main analysis period:
2011 to 2021

Optional validation file:
Combined_Tableau_Analysis_2011_2021.csv
